In [1]:
# Notebook cell: imports
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

In [2]:
# Notebook cell: config (edit these)
DATASET_PATH = "/pbs/home/s/smartinez/ML4CollEffects/data/neural/neural_xsuite_dataset_2026-04-28T09:46:07.npz"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LATENT_DIM = 64
BATCH_SIZE = 2
EPOCHS = 20
LR = 1e-3

# If training on CPU, set this smaller (e.g. 1024 or 2048). If CUDA, you can use 8192.
N_POINTS_USED = 2048   # <= change to 8192 if GPU and enough memory/time

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
# Notebook cell: dataset + optional subsampling to N_POINTS_USED
class CloudDataset(Dataset):
    def __init__(self, npz_path: str, split: str = "train", n_points_used: int | None = None):
        raw = np.load(npz_path, allow_pickle=True)
        idx = raw[split]
        self.x = raw["X_cloud"][idx]  # [Ns, Np, 6]
        self.n_points_used = n_points_used

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, i):
        x = self.x[i].astype(np.float32)  # [Np,6]
        if self.n_points_used is not None and self.n_points_used < x.shape[0]:
            # random subsample points (keeps it a set)
            sel = np.random.choice(x.shape[0], self.n_points_used, replace=False)
            x = x[sel]
        return torch.from_numpy(x).float()

def get_n_points(npz_path: str):
    raw = np.load(npz_path, allow_pickle=True)
    return raw["X_cloud"].shape[1]

NP_FULL = get_n_points(DATASET_PATH)
print("Full points per cloud in dataset:", NP_FULL)
print("Training points used per cloud:", N_POINTS_USED)

Full points per cloud in dataset: 8192
Training points used per cloud: 2048


In [4]:
# Notebook cell: Chamfer loss
def chamfer_l2(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    # a,b: [B,N,6]
    dist = torch.cdist(a, b, p=2) ** 2
    min_a = dist.min(dim=2).values
    min_b = dist.min(dim=1).values
    return (min_a.mean() + min_b.mean())

In [5]:
# Notebook cell: model (CloudAE)
class CloudEncoder(nn.Module):
    def __init__(self, in_dim=6, hidden=128, latent_dim=64):
        super().__init__()
        self.phi = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
        )
        self.rho = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, latent_dim),
        )

    def forward(self, x):
        # x: [B,N,6]
        h = self.phi(x)        # [B,N,H]
        h = h.mean(dim=1)      # [B,H]
        z = self.rho(h)        # [B,latent]
        return z

class CloudDecoder(nn.Module):
    def __init__(self, latent_dim=64, n_points=2048, hidden=256, out_dim=6):
        super().__init__()
        self.n_points = n_points
        self.mlp = nn.Sequential(
            nn.Linear(latent_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, n_points * out_dim),
        )

    def forward(self, z):
        x = self.mlp(z)
        return x.view(z.shape[0], self.n_points, 6)

class CloudAE(nn.Module):
    def __init__(self, n_points, latent_dim=64):
        super().__init__()
        self.enc = CloudEncoder(latent_dim=latent_dim)
        self.dec = CloudDecoder(latent_dim=latent_dim, n_points=n_points)

    def forward(self, x):
        z = self.enc(x)
        x_hat = self.dec(z)
        return x_hat, z

model = CloudAE(n_points=N_POINTS_USED, latent_dim=LATENT_DIM).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-6)
print("device:", DEVICE)

device: cpu


In [6]:
# Notebook cell: loaders
train_loader = DataLoader(
    CloudDataset(DATASET_PATH, "train", n_points_used=N_POINTS_USED),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)
val_loader = DataLoader(
    CloudDataset(DATASET_PATH, "val", n_points_used=N_POINTS_USED),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

In [ ]:
# Notebook cell: train + validate
@torch.no_grad()
def eval_loss(loader):
    model.eval()
    s = 0.0
    n = 0
    for x in loader:
        x = x.to(DEVICE)
        x_hat, _ = model(x)
        loss = chamfer_l2(x_hat, x)
        b = x.size(0)
        s += loss.item() * b
        n += b
    return s / max(1, n)

for epoch in range(1, EPOCHS + 1):
    model.train()
    s = 0.0
    n = 0
    for x in train_loader:
        x = x.to(DEVICE)
        x_hat, z = model(x)
        loss = chamfer_l2(x_hat, x)
        opt.zero_grad()
        loss.backward()
        opt.step()
        b = x.size(0)
        s += loss.item() * b
        n += b

    tr = s / max(1, n)
    va = eval_loss(val_loader)

    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch={epoch:03d} train_chamfer={tr:.4e} val_chamfer={va:.4e}")

In [ ]:
# Notebook cell: extract latents (train/val) for visualization
@torch.no_grad()
def collect_latents(split: str, max_batches: int | None = 50):
    loader = DataLoader(
        CloudDataset(DATASET_PATH, split, n_points_used=N_POINTS_USED),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )
    model.eval()
    zs = []
    for bi, x in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break
        x = x.to(DEVICE)
        _, z = model(x)
        zs.append(z.cpu().numpy())
    return np.concatenate(zs, axis=0)

Z_train = collect_latents("train", max_batches=50)
Z_val = collect_latents("val", max_batches=50)

print("Z_train:", Z_train.shape, "mean/std:", Z_train.mean(), Z_train.std())
print("Z_val  :", Z_val.shape, "mean/std:", Z_val.mean(), Z_val.std())